- Perform merge in this layer for incremental data loading
- Add a new column called processed_date
- Customer_name is uppercased

In [0]:
spark.sql("""
        select *,
                upper(customer_name) as customer_name_upper,
                current_timestamp() as Timestamp_Current,
                date(current_timestamp()) as process_date
        from datamodeling.bronze.bronze_table
          """).createOrReplaceTempView("silver_source")

#### Merge Statement

In [0]:
%sql
select * from 
silver_source

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,product_price,quantity,revenue,country,last_updated,customer_name_upper,Timestamp_Current,process_date
1005,2026-05-05,505,Michael Brown,michael.brown@gmail.com,2005,Wireless Headphones,Electronics,129.99,1,129.99,Canada,2026-05-08,MICHAEL BROWN,2026-05-08T07:41:11.228Z,2026-05-08
1006,2026-05-06,506,Emma Wilson,emma.wilson@yahoo.com,2006,Coffee Maker,Home Appliances,89.50,1,89.50,United Kingdom,2026-05-08,EMMA WILSON,2026-05-08T07:41:11.228Z,2026-05-08


In [0]:
spark.sql("create schema if not exists datamodeling.silver")

DataFrame[]

In [0]:
if spark.catalog.tableExists('datamodeling.silver.silver_table'):
    spark.sql("""
              merge into datamodeling.silver.silver_table as dest
              using silver_source as src
              on dest.order_id = src.order_id
              when matched then update set *
              when not matched then insert *
              """)
else:
    spark.sql("""
              create table if not exists datamodeling.silver.silver_table
              using delta
              as
              select *
              from silver_source
              """)

In [0]:
%sql
select * from datamodeling.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,product_price,quantity,revenue,country,last_updated,customer_name_upper,Timestamp_Current,process_date
1001,2026-05-01,501,John Carter,john.carter@gmail.com,2001,Wireless Mouse,Electronics,25.99,2,51.98,USA,2026-05-07,JOHN CARTER,2026-05-08T06:42:31.016Z,2026-05-08
1002,2026-05-02,502,Emma Watson,emma.watson@yahoo.com,2002,Office Chair,Furniture,149.50,1,149.50,Canada,2026-05-07,EMMA WATSON,2026-05-08T06:42:31.016Z,2026-05-08
1003,2026-05-03,503,Rahul Sharma,rahul.sharma@gmail.com,2003,Mechanical Keyboard,Electronics,89.99,3,269.97,India,2026-05-07,RAHUL SHARMA,2026-05-08T06:42:31.016Z,2026-05-08
1004,2026-05-04,504,Sophia Lee,sophia.lee@outlook.com,2004,Running Shoes,Sports,79.99,2,159.98,Australia,2026-05-07,SOPHIA LEE,2026-05-08T06:42:31.016Z,2026-05-08


In [0]:
spark.sql("SHOW CATALOGS").display()
spark.sql("SHOW SCHEMAS IN datamodeling").display()
spark.sql("SHOW TABLES IN datamodeling.silver").display()

catalog
datamodeling
samples
system
workspace


databaseName
bronze
default
information_schema
silver


database,tableName,isTemporary
silver,silver_table,false
,silver_source,true


### Same process but using PySpark

In [0]:
%sql
drop table datamodeling.silver.silver_table

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# Reading the bronze table
silver_source = spark.table("datamodeling.bronze.bronze_table")

# Applying the transformations
silver_df = (
    silver_source
    .withColumn("customer_name_upper", upper("customer_name"))
    .withColumn("Timestamp_Current", current_timestamp())
    .withColumn("process_date", current_date())
)

# Ceating schema if it doesnot exist
spark.sql("create schema if not exists datamodeling.silver")

# Name of the target table
target_table = "datamodeling.silver.silver_table"

# Cheking if the table already exists
if spark.catalog.tableExists(target_table):

    delta_table = DeltaTable.forName(spark, target_table)

    # Performing merge(upsert)
    (
        delta_table.alias("dest")
        .merge(
            silver_df.alias("src"),
            "dest.order_id = src.order_id"
        )
         .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    # Creating table for the first load
    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )



In [0]:
%sql
select * from datamodeling.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,product_price,quantity,revenue,country,last_updated,customer_name_upper,Timestamp_Current,process_date
1001,2026-05-01,501,John Carter,john.carter@gmail.com,2001,Wireless Mouse,Electronics,25.99,2,51.98,USA,2026-05-07,JOHN CARTER,2026-05-08T06:42:31.016Z,2026-05-08
1002,2026-05-02,502,Emma Watson,emma.watson@yahoo.com,2002,Office Chair,Furniture,149.50,1,149.50,Canada,2026-05-07,EMMA WATSON,2026-05-08T06:42:31.016Z,2026-05-08
1003,2026-05-03,503,Rahul Sharma,rahul.sharma@gmail.com,2003,Mechanical Keyboard,Electronics,89.99,3,269.97,India,2026-05-07,RAHUL SHARMA,2026-05-08T06:42:31.016Z,2026-05-08
1004,2026-05-04,504,Sophia Lee,sophia.lee@outlook.com,2004,Running Shoes,Sports,79.99,2,159.98,Australia,2026-05-07,SOPHIA LEE,2026-05-08T06:42:31.016Z,2026-05-08
1005,2026-05-05,505,Michael Brown,michael.brown@gmail.com,2005,Wireless Headphones,Electronics,129.99,1,129.99,Canada,2026-05-08,MICHAEL BROWN,2026-05-08T07:39:31.372Z,2026-05-08
1006,2026-05-06,506,Emma Wilson,emma.wilson@yahoo.com,2006,Coffee Maker,Home Appliances,89.50,1,89.50,United Kingdom,2026-05-08,EMMA WILSON,2026-05-08T07:39:31.372Z,2026-05-08
